In [3]:
from __future__ import print_function, division
import os
import numpy as np
import math
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# --------------------------------------------------
# DIRECTORIES
# --------------------------------------------------
SIM_DIR = "/home/hp/raytrace_work/raytrace_results/Simulation_Data_201x201"
OUT_DIR = "/home/hp/raytrace_work/raytrace_results/Analysis_201x201"

if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)

# --------------------------------------------------
# LOAD PARAMETERS
# --------------------------------------------------
params    = np.load(os.path.join(SIM_DIR, "sim_params.npy"),
                    allow_pickle=True).item()
freq_list = np.array(params['freq_list'])
ny        = int(params['ny'])
nx        = int(params['nx'])
rect      = params['rect']
rsph      = params['rsph']
subset_A  = params['subset_A']   # 25 ray global indices

def mhz_int(f):
    return int(round(f / 1.e6))

# --------------------------------------------------
# PHYSICAL Y,Z COORDINATES
# --------------------------------------------------
y_arr = np.linspace(rect[0], rect[2], nx)
z_arr = np.linspace(rect[1], rect[3], ny)

# --------------------------------------------------
# CRITICAL ELECTRON DENSITY (paper eq 3, CGS)
# Ne_cr = me * omega^2 / (4 * pi * e^2)
# --------------------------------------------------
me    = 9.10938e-28
e_cgs = 4.80326e-10
pi    = math.pi
omega = 2.0 * pi * freq_list
Ne_cr = (me * omega**2) / (4.0 * pi * e_cgs**2)

In [4]:
# -------------------------------------------------------------
# SIMULATION REFLECTION FOR FIXED IMPACT PARAMETER vs FREQUENCY
# -------------------------------------------------------------

iy_fixed = 55
iz_fixed = 55

Y_fixed  = y_arr[iy_fixed]
Z_fixed  = z_arr[iz_fixed]
b_fixed  = math.sqrt(Y_fixed**2 + Z_fixed**2)

# global index in flattened ray array
global_idx = iy_fixed * ny + iz_fixed

print("Fixed ray:")
print("  iy={} Y={:.6f} Rs".format(iy_fixed, Y_fixed))
print("  iz={} Z={:.6f} Rs".format(iz_fixed, Z_fixed))
print("  b ={:.6f} Rs".format(b_fixed))
print("  global_idx = {}".format(global_idx))

# --------------------------------------------------
# FOR EACH FREQUENCY:
# 1. Load trajectory for this ray
# 2. Compute R_refl = min(r) along trajectory
# --------------------------------------------------
mhz_arr        = []
R_crit_sim     = []

print("\n{:>8}  {:>12}  {:>12}".format(
    "MHz", "Ne_cr", "R_refl Rs"))
print("-" * 38)

for freq_hz in freq_list:
    mhz   = mhz_int(freq_hz)
    omega = 2.0 * np.pi * freq_hz
    Ne_cr = (me * omega**2) / (4.0 * pi * e_cgs**2)

    # load trajectory for this frequency
    traj_path = os.path.join(
        SIM_DIR, "traj_{}mhz.npy".format(mhz))

    if not os.path.exists(traj_path):
        print("{:>8}  FILE NOT FOUND".format(mhz))
        mhz_arr.append(mhz)
        R_crit_sim.append(np.nan)
        continue

    traj_all = np.load(traj_path)

    # extract single ray trajectory
    ray_traj = traj_all[global_idx, :, :]

    # compute heliocentric distance at each step
    r_trace = np.sqrt(ray_traj[:, 0]**2 +
                      ray_traj[:, 1]**2 +
                      ray_traj[:, 2]**2)

    # ignore zero-padded steps
    nonzero = r_trace > 0.01

    if nonzero.sum() > 0:
        R_refl = r_trace[nonzero].min()
    else:
        R_refl = np.nan

    mhz_arr.append(mhz)
    R_crit_sim.append(R_refl)

    print("{:>8}  {:>12.4e}  {:>12}".format(
        mhz, Ne_cr,
        "{:.6f}".format(R_refl)
        if not np.isnan(R_refl) else "NOT FOUND"))

    del traj_all

mhz_arr    = np.array(mhz_arr,    dtype=float)
R_crit_sim = np.array(R_crit_sim, dtype=float)

valid = ~np.isnan(R_crit_sim)

Fixed ray:
  iy=55 Y=-0.900000 Rs
  iz=55 Z=-0.900000 Rs
  b =1.272792 Rs
  global_idx = 11110

     MHz         Ne_cr     R_refl Rs
--------------------------------------


/home/hp/miniconda3/envs/py27/lib/python2.7/site-packages/ipykernel_launcher.py:59: RuntimeWarning: invalid value encountered in greater


      15    2.7909e+06      2.026295
      30    1.1164e+07      1.664039
      45    2.5118e+07      1.518873
      60    4.4655e+07      1.442272
      75    6.9773e+07      1.396131
      90    1.0047e+08      1.365974
     105    1.3676e+08      1.345118
     120    1.7862e+08      1.330087
     135    2.2607e+08      1.318914
     150    2.7909e+08      1.310382
     165    3.3770e+08      1.303726
     180    4.0189e+08      1.298450
     195    4.7167e+08      1.294197
     210    5.4702e+08      1.290720
     225    6.2796e+08      1.287846
     240    7.1448e+08      1.285455
     255    8.0658e+08      1.283421
     270    9.0426e+08      1.281708
     285    1.0075e+09      1.280222
     300    1.1164e+09      1.278950


In [5]:
# --------------------------------------------------
# PLOT: R_crit vs Frequency
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(mhz_arr[valid], R_crit_sim[valid],
        linestyle=u'-', marker=u'o',
        markersize=6, linewidth=1.5,
        color=u'blue',
        label=u'Simulation R_crit')

ax.axhline(1.0, color=u'orange',
           linestyle=u'--', linewidth=2.0,
           label=u'Sun surface (R=1 Rs)')

ax.set_xlabel(u'Frequency (MHz)', fontsize=12)
ax.set_ylabel(u'Critical Radius (Solar Radii)',
              fontsize=12)
ax.set_title(
    u'Simulation Critical Radius vs Frequency\n'
    u'Fixed ray: iy={} Y={:.4f} Rs, '
    u'iz={} Z={:.4f} Rs, b={:.4f} Rs'.format(
        iy_fixed, Y_fixed,
        iz_fixed, Z_fixed, b_fixed),
    fontsize=11)
ax.set_xticks(mhz_arr)
ax.set_xticklabels([str(int(m)) for m in mhz_arr],
                    rotation=45, fontsize=8)
ax.legend(fontsize=10)
ax.grid(True, linestyle=u'--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(
    OUT_DIR,
    "R_crit_sim_vs_freq_iy{}_iz{}.png".format(
        iy_fixed, iz_fixed)), dpi=150)
plt.close()
print("\nSaved Plot: R_crit vs frequency.")


Saved Plot: R_crit vs frequency.
